In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

# 1 bert-fa-base-uncased-sentiment-snappfood

https://huggingface.co/HooshvareLab/bert-fa-base-uncased-sentiment-snappfood

In [ ]:
# Load model directly if you don't download it before

# tokenizer = AutoTokenizer.from_pretrained("HooshvareLab/bert-fa-base-uncased-sentiment-snappfood")
# model = AutoModelForSequenceClassification.from_pretrained("HooshvareLab/bert-fa-base-uncased-sentiment-snappfood")
# # save model
# save_directory = "../models/HooshvareLab/sentiment-snappfood"

# tokenizer.save_pretrained(save_directory)
# model.save_pretrained(save_directory)

# device = "cuda" if torch.cuda.is_available() else "cpu"
# model.to(device)

In [ ]:
# load it if you have downloaded it
model_path = os.getenv('MODELS')
save_directory = f"{model_path}/HooshvareLab/sentiment-snappfood"

tokenizer = AutoTokenizer.from_pretrained(save_directory)
model = AutoModelForSequenceClassification.from_pretrained(save_directory)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval() 

In [ ]:
data_path = os.getenv('DATA_PATH')
# all_comments = pd.read_csv(f'{data_path}/all_labeled_comments.csv')
train_data = pd.read_csv(f"{data_path}/30k_labeled_comments.csv")

In [ ]:
sample = train_data[train_data['sentiment']=='Negative'].sample(1000).copy()

In [ ]:
labels = ["Positive", "Negative"]

def batch_predict(texts, batch_size=64):
    results = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        
        inputs = tokenizer(batch_texts, return_tensors="pt", truncation=True, padding=True, max_length=512)
        inputs = {key: val.to(device) for key, val in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)

        threshold = 0.9  
        predicted_classes = (probs[:, 0] < threshold).cpu().numpy().astype(int)  
        batch_labels = [labels[idx] for idx in predicted_classes]
        
        results.extend(batch_labels)
    
    return results

comments = sample["cleaned_comment"].tolist()
predicted_labels = batch_predict(comments, batch_size=128)
sample["sentiment_by_parsbert"] = predicted_labels

print("✅ پردازش تمام شد! نتایج ذخیره شدند.")

In [ ]:
sample[sample['sentiment_by_parsbert']=='Negative'].shape

# 2 trained model

In [ ]:
data_path = os.getenv('DATA_PATH')
all_comments = pd.read_csv(f'{data_path}/all_labeled_comments.csv')
train_data = pd.read_csv(f"{data_path}/labeled_comments.csv")

In [ ]:
print(torch.cuda.is_available())  
print(torch.cuda.get_device_name(0))  

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = Tokenizer(num_words=5000, lower=True, split=' ')
tokenizer.fit_on_texts(train_data['cleaned_comment'].values)

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim * 2, 1)

    def forward(self, lstm_output):
        attn_weights = torch.tanh(self.attention(lstm_output)).squeeze(-1)
        attn_weights = F.softmax(attn_weights, dim=1).unsqueeze(1)
        context_vector = torch.bmm(attn_weights, lstm_output).squeeze(1)
        return context_vector

class AdvancedBiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, dropout):
        super(AdvancedBiLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=4, bidirectional=True, batch_first=True, dropout=dropout)
        self.attention = Attention(hidden_dim)
        self.batch_norm1 = nn.BatchNorm1d(hidden_dim * 2)
        self.fc1 = nn.Linear(hidden_dim * 2, 512)
        self.batch_norm2 = nn.BatchNorm1d(512)
        self.fc2 = nn.Linear(512, 256)
        self.batch_norm3 = nn.BatchNorm1d(256)
        self.fc3 = nn.Linear(256, 128)
        self.dropout = nn.Dropout(dropout)
        self.fc4 = nn.Linear(128, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.lstm(x)
        x = self.attention(x)  
        x = self.batch_norm1(x)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.batch_norm2(x)
        x = self.dropout(F.relu(self.fc2(x)))
        x = self.batch_norm3(x)
        x = self.dropout(F.relu(self.fc3(x)))
        x = self.fc4(x)
        return x
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AdvancedBiLSTM(vocab_size=5000, embed_dim=300, hidden_dim=256, output_dim=3, dropout=0.5)
model_path = os.getenv('MODELS')
state_dict = torch.load(f'{model_path}/sentiment_analyse_model/sentiment_analysis_model_on_30K_comment.pth', map_location=device)
model.load_state_dict(state_dict)

model = model.to(device)

model.eval()

In [ ]:
label_mapping = {
    0: 'Negative', 
    1: 'Netural',
    2: 'Positive'
}

In [ ]:
comments = train_data.sample(10)
unlabeled_sequences = tokenizer.texts_to_sequences(comments['cleaned_comment'].values)
unlabeled_padded = pad_sequences(unlabeled_sequences, maxlen=2048)
unlabeled_tensor = torch.tensor(unlabeled_padded, dtype=torch.long).to(device)

unlabeled_tensor = torch.tensor(unlabeled_padded, dtype=torch.long).to(device)

batch_size = 1024  # 
dataset = TensorDataset(unlabeled_tensor)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

all_predictions = []

model.eval()
with torch.no_grad():
    for batch in dataloader:
        inputs = batch[0]
        outputs = model(inputs)
        print(outputs)
        preds = torch.argmax(outputs, dim=1)
        all_predictions.extend(preds.cpu().numpy())

all_predictions = np.array(all_predictions)
comments['sentiment_by_trained_model'] = [label_mapping[pred] for pred in all_predictions]

comments.head()